# Vaultify R1 Control Panel — Clean

This notebook is **operational only**. Application logic lives under `src/vaultify/`.

Current checkpoint:
- R1 Step 18: CLOSED / PASS
- R1 Step 19: IN PROGRESS
- Next gate: Step 19 real tokenizer + Docling availability

Rules:
- Work only from `release/v0.1-extraction`.
- Do not paste historical Step 6–18 implementation/regression cells here.
- GitHub modules + committed tests are the source of the extracted implementation.
- Run cells from top to bottom after every fresh Colab runtime.


In [ ]:
# CELL 1 — Fresh Colab dependencies

%pip install -q \
    flask \
    flask-sqlalchemy \
    flask-login \
    flask-wtf \
    pytest \
    sentence-transformers \
    qdrant-client \
    groq \
    filetype \
    docling

print("✅ Vaultify R1 fresh-runtime dependencies ready.")


In [ ]:
# CELL 2 — Clone or update the release branch

from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/IhabAltekreeti/vaultify.git"
BRANCH = "release/v0.1-extraction"
REPO_DIR = Path("/content/vaultify-r1")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "-q", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"],
        check=True,
    )

os.chdir(REPO_DIR)

head = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

print(f"✅ Vaultify ready at: {REPO_DIR}")
print(f"✅ Branch: {BRANCH}")
print(f"✅ HEAD: {head}")


In [ ]:
# CELL 3 — Python path + package verification

import sys

SRC_DIR = str(REPO_DIR / "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from vaultify import config

assert config.PROJECT_NAME == "Vaultify"
assert config.COLLECTION_NAME == "vaultify_v3_documents"
assert config.VECTOR_SIZE == 384

print("✅ Extracted Vaultify package imports successfully.")
print(f"✅ Collection: {config.COLLECTION_NAME}")
print(f"✅ Vector size: {config.VECTOR_SIZE}")


In [ ]:
# CELL 4 — Full committed regression suite
# If anything fails, this cell prints the exact failing test automatically.

import os
import subprocess

result = subprocess.run(
    [
        "pytest",
        "-vv",
        "tests/regression",
        "-x",
        "--tb=short",
    ],
    cwd=REPO_DIR,
    env={
        **os.environ,
        "PYTHONPATH": SRC_DIR,
    },
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr.strip():
    print("PYTEST STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "R1 regression suite failed. "
        "Use the FAILED/ERROR section printed above; do not continue to later cells."
    )

print("✅ R1 extracted regression suite PASS.")


In [ ]:
# CELL 5 — Show the compact release checkpoint

release_state = (REPO_DIR / "RELEASE_STATE.md").read_text(encoding="utf-8")
print(release_state)


In [ ]:
# CELL 6 — Live cloud/runtime health check
# Read-only: verifies secrets, Qdrant, Groq, and the embedding service.

from google.colab import userdata

from vaultify.config import COLLECTION_NAME, LLM_MODEL
from vaultify.services.embeddings import EmbeddingService
from vaultify.services.llm import create_groq_client, probe_groq_connection
from vaultify.services.qdrant import create_qdrant_client, list_collection_names

QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

assert QDRANT_URL, "Add QDRANT_URL to Colab Secrets."
assert QDRANT_API_KEY, "Add QDRANT_API_KEY to Colab Secrets."
assert GROQ_API_KEY, "Add GROQ_API_KEY to Colab Secrets."

qdrant = create_qdrant_client(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

collection_names = list_collection_names(qdrant)
assert COLLECTION_NAME in collection_names, {
    "expected": COLLECTION_NAME,
    "available": collection_names,
}

groq_client = create_groq_client(api_key=GROQ_API_KEY)
probe_response = probe_groq_connection(groq_client)
assert probe_response

embedding_service = EmbeddingService()

print("✅ Qdrant Cloud connection PASS.")
print(f"✅ Collection found: {COLLECTION_NAME}")
print("✅ Groq connection PASS.")
print(f"✅ Groq model: {LLM_MODEL}")
print("✅ Embedding service ready.")
print("✅ No secret value was printed.")


In [ ]:
# CELL 7 — R1 Step 19 final live gate
# Canonical V2 chunker with the REAL tokenizer + Docling availability.
# This cell does NOT write to Qdrant.

import importlib
import re

import vaultify.services.ingestion as ingestion

importlib.invalidate_caches()
ingestion = importlib.reload(ingestion)

chunker = ingestion.CanonicalChunkerV2(embedding_service.model)

assert chunker.max_chunk_tokens == 240

long_text = " ".join(["financial-data"] * 900)
huge_table_row = "| Oversized row | " + " ".join(["123456"] * 700) + " |"

synthetic_markdown = "\n".join(
    [
        "# Synthetic Safety Test",
        "",
        long_text,
        "",
        "| Metric | 2025 | 2024 |",
        "|---|---|---|",
        "| Total net sales | 416,161 | 391,035 |",
        huge_table_row,
    ]
)

chunks = chunker.chunk_markdown(synthetic_markdown)
assert chunks

token_counts = [
    chunker.count_embedding_tokens(chunk["text"])
    for chunk in chunks
]

assert max(token_counts) <= 240
assert [chunk["chunk_index"] for chunk in chunks] == list(range(len(chunks)))

normalized_texts = [
    re.sub(r"\s+", " ", chunk["text"]).strip()
    for chunk in chunks
]

assert len(normalized_texts) == len(set(normalized_texts))
assert any(chunk["chunk_type"] == "table" for chunk in chunks)
assert all("|---|" not in chunk["text"] for chunk in chunks)

validated = ingestion.validate_pdf_upload(
    "Example Financial Report.pdf",
    b"%PDF-1.7\nVaultify synthetic PDF regression",
)

assert validated.safe_filename == "Example_Financial_Report.pdf"
assert len(validated.document_hash) == 64

point_a = ingestion.deterministic_point_id("tenant_demo", "a" * 64, 7)
point_b = ingestion.deterministic_point_id("tenant_demo", "a" * 64, 7)
point_other = ingestion.deterministic_point_id("tenant_other", "a" * 64, 7)

assert point_a == point_b
assert point_a != point_other

converter = ingestion.build_document_converter()
assert converter is not None

print("✅ R1 STEP 19 PASS")
print("✅ Canonical V2 ingestion core works with the real tokenizer.")
print(f"✅ Maximum generated embedding tokens: {max(token_counts)} / 240")
print("✅ Oversized text/table rows split safely.")
print("✅ Markdown separator rows are removed.")
print("✅ Exact duplicate protection is active.")
print("✅ PDF validation + SHA-256 hashing passed.")
print("✅ Tenant/document/chunk point IDs are deterministic.")
print("✅ Docling converter is available.")
print("✅ No live Qdrant points were created, updated, or deleted.")


## Stop here

If Cell 7 prints `✅ R1 STEP 19 PASS`, Step 19 can be closed and recorded in `RELEASE_STATE.md`.

Do **not** paste old Step 6–18 cells back into this notebook. Their behavior is preserved in `src/vaultify/` and `tests/regression/`.

The next bounded unit after Step 19 is the upload/document-management web slice.
